The idea of gradient accumalation is simple, think of it as normal gradient descent, but we take the average of $N$ gradients before updating the parameters with the optimizer.

This way we can train on an effective larger batch size without having to increase in memory, it can be combined with activation re-computation too.

- But no, gradient accumalation doesn't save memory

And it's really simple to implement in code! We'll put down first a simple, original training loop here.

In [9]:
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
import mini_gpt

dataset = load_dataset("imdb", split="train[:500]") 
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")
dataset = dataset.map(tokenize_fn, batched=True)
dataset.set_format("torch", columns=["input_ids"])

config = mini_gpt.ModelConfig()
model = mini_gpt.MiniTransformer(config)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
loss_fn = torch.nn.CrossEntropyLoss()

dataloader = DataLoader(dataset, batch_size=8)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

without gradient accumalation.

In [10]:
for epoch in range(1):
    optimizer.zero_grad()
    
    for i, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        input_ids = torch.clamp(input_ids, max=config.vocab_size - 1) # clip to [0, 999], as our model has 1000 vocab
        
        logits = model(input_ids)
        loss = loss_fn(logits.view(-1, config.vocab_size), input_ids.view(-1))
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        
        if i % 8 == 0:
            print(f"Step {i}, Loss: {loss.item():.4f}")
    
    print(f"Epoch {epoch} done")

Step 0, Loss: 6.8150
Step 8, Loss: 3.3114
Step 16, Loss: 2.8699
Step 24, Loss: 1.9913
Step 32, Loss: 1.8595
Step 40, Loss: 1.6677
Step 48, Loss: 1.4718
Step 56, Loss: 1.4673
Epoch 0 done


with gradient accumalation of step 8, we effectively have a batch size of 8 * 8 = 64

In [11]:
config = mini_gpt.ModelConfig()
model = mini_gpt.MiniTransformer(config)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
loss_fn = torch.nn.CrossEntropyLoss()

In [12]:
accumulation_steps = 8

for epoch in range(1):
    optimizer.zero_grad()
    
    for i, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        input_ids = torch.clamp(input_ids, max=config.vocab_size - 1) # clip to [0, 999], as our model has 1000 vocab
        
        logits = model(input_ids)
        loss = loss_fn(logits.view(-1, config.vocab_size), input_ids.view(-1))
        loss.backward()

        # gradient accumalation
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        if i % 8 == 0:
            print(f"Step {i}, Loss: {loss.item():.4f}")
    
    print(f"Epoch {epoch} done")

Step 0, Loss: 6.9343
Step 8, Loss: 5.9615
Step 16, Loss: 5.3357
Step 24, Loss: 4.3672
Step 32, Loss: 4.1262
Step 40, Loss: 3.8968
Step 48, Loss: 3.7119
Step 56, Loss: 3.7127
Epoch 0 done


Gradient accumalation does come with the downside that training is slower, since you you are increasing the batch size, but having fewer updates:

- Without GA: 500/8 = 62 batches → 62 optimizer steps
- With GA (steps=8): 500/8 = 62 batches → only 62/8 = ~8 optimizer steps
